In [1]:
import re 
import pandas as pd
# carica la libreria Pandas e le dà il soprannome pd
df = pd.read_csv("repubblica.csv")
# apre il file repubblica.csv e lo trasforma in un DataFrame
# il risultato è salvato in nella variabile df
# tutto il contenuto del CSV è caricato dentro df
print(df.columns)
#df.columns è un attributo del DataFrame che contiene l'elenco dei nomi delle sue colonne

def tokenizza(t):
    return re.findall(r"\w+", str(t).lower())
# definisco cosa fa la funzione tokenizza 
# str(t) prende qualunque valore del testo in una cella e lo trasforma in una stringa di testo
# lower() trasforma tutte le lettere maiuscole in minuscole
# senza lower, parole identiche ma scritte diversamente verrebbero trattate come token distinti
df["token"] = df["full_text"].apply(tokenizza) 
# il risultato è salavato in una colonna nuova chiamata token
# il DataFrame ha un colonna in più
# questa riga tokenizza l'intero dataset, non solo il primo articolo
print(df["full_text"].iloc[0])   # il testo originale del primo articolo
print(df["token"].iloc[0])       # la sua versione tokenizzata
# iloc[0] serve a selezionare una riga specifica del DataFrame, 
# indicandola per posizione numerica, partendo da 0 e non da 1
# è solo una prova per vedere cosa succede
print(len(df))                      # quante righe ha il dataset in totale 63701
print(df["token"].notna().sum())    # quante di quelle righe hanno un valore in "token"
#questo mi serve per avere la conferma che la tokenizzazione è avvenuta per tutte le celle del testo

from collections import Counter 
# conto le occorrenze per capire se ha senso fare la ricerca su quei paesi
all_tokens = []
for lista in df["token"]:       # prendi ogni riga della colonna "token"
    for tok_lista in lista:           # e di quella riga esamina ogni parola
    # la colonna token è una lista di liste
    # per arrivare alle singole parole bisogna prima entrare in ogni articolo
    # poi dentro di esso scorrere ogni parola
        all_tokens.append(tok_lista)
        # contiene tutte le parole di tutti gli articoli in un'unica lista semplice
        # non più divisa per articolo)
occorrenze = Counter(all_tokens)

print(occorrenze["siria"]) #6489  
print(occorrenze["afghanistan"]) #1603
print(occorrenze["iraq"]) # 3707
print(occorrenze["sudan"]) #1598
print(occorrenze["libia"]) #4322
print(occorrenze["siriano"]) #1745
print(occorrenze["siriana"]) #933
print(occorrenze["siriani"]) #1798
print(occorrenze["siriane"]) #392
print(occorrenze["afghano"]) #130
print(occorrenze["afghana"]) #94
print(occorrenze["afghani"]) #162
print(occorrenze["afghane"]) #65
print(occorrenze["iracheno"]) #890
print(occorrenze["irachena"]) #393
print(occorrenze["iracheni"]) #487
print(occorrenze["irachene"]) #402
print(occorrenze["sudanese"]) #173
print(occorrenze["sudanesi"]) #202
print(occorrenze["libico"]) #761
print(occorrenze["libica"]) #830
print(occorrenze["libici"]) #571
print(occorrenze["libiche"]) #598
print(occorrenze["2014"]) #9426

#co-occorrenze per un'analisi semantica
parole_siria = {"siria", "siriano", "siriana", "siriani", "siriane"}
parole_arresto = {"arresto", "arresti", "arrestato", "arrestata", "arrestati", "arrestate"}
# creo il mio set di parole non in ordine che voglio cercare
def vicine_in_lista(lista, distanza=100):
    pos_siria = [pos_lista for pos_lista, tok_lista in enumerate(lista) if tok_lista in parole_siria] #list comprehension, un modo compatto di scrivere, su una sola riga, un ciclo for che costruisce una nuova lista
    # for pos_lista, tok_lista in enumerate(lista); questo è il loop che scorre la lista e controlla la posizione delle parole del set
    # if tok_lista in parole_siria; per ogni giro del ciclo controlla questa condizione
    pos_arresto = [pos_lista for pos_lista, tok_lista in enumerate(lista) if tok_lista in parole_arresto] #list comprehension
    return any(abs(pos_lista1 - pos_lista2) <= distanza for pos_lista1 in pos_siria for pos_lista2 in pos_arresto) 
    #ogni posizione del set viene confrontata con ogni posizione di arresto
    # pos_lista1 - pos_lista2 ; per ogni posizone calcola la differenza, che deve essere ≤ 5, producendo true o false
    # abs() prende quel numero e lo rende sempre positivo, perché non interessa cosa viene prima o dopo, solo quanto sono lontane
df["co_occorrenza"] = df["token"].apply(vicine_in_lista)
# la f è applicata a tutta la lista token e il risultato è salvato in una nuova colonna chiamata co-occorrenza
print(df["co_occorrenza"].sum())
# conta quante celle della colonna hanno valore True, e aggiunge 1

Index(['headline', 'full_text', 'url'], dtype='str')
Nella lunga e complicata vicenda delle adozioni in Congo c'è un'ombra pesante: alcuni dei bambini arrivati in Italia negli ultimi anni non sarebbero stati adottabili, perché avevano una famiglia naturale alla quale sarebbero stati sottratti con l'inganno e, presumibilmente, dietro pagamento di somme di denaro. A denunciare questa vicenda è stata oggi la vicepresidente della Commissione adozioni internazionali (Cai), Silvia Della Monica, nel corso di un convegno promosso dal senatore Maurizio Romani (Idv/Gruppo misto). Della Monica ha spiegato che sulla vicenda la Commissione ha avviato accertamenti e che "è stata interessata l'Autorità giudiziaria". "Se i bambini sono stati strappati alle famiglie e gli enti ne erano consapevoli, ci sono precise responsabilità da accertare" ha detto. Ha poi ricordato che di questi fatti si era occupato il settimanale L'Espresso nel luglio scorso , in un'inchiesta che citava l'ente accreditato per le 

Ora provo a dividere il testo in frasi e capire se i due set di parole sono presenti nella stessa frase.

In [2]:
import re 
import pandas as pd

df = pd.read_csv("repubblica.csv")
df = df.head(1000)
print(df.columns)

def tokenizza(t):
    return re.findall(r"\w+", str(t).lower())


def occorenza_siria_arresto(full_text, parole_siria, parole_arresto):
    frasi = re.split(r"(?<=[.!?])\s+", str(full_text))
    # divido il testo originale in una lista di frasi
    # taglio dopo un punto, punto esclamativo o interrogativo seguito da uno spazio
    frasi_trovate = []
    for frase in frasi: #ogno ciclo scorre la lista divisa in frasi sopra, e assegna la variabile frase a ogni elemento
        token_frase = set(re.findall(r"\w+", frase.lower()))
        # la re è la tokenizzazione, essenzialmente
        # trasformo la lista in un set, cioè un insieme senza duplicati e senza ordine
        if token_frase & parole_siria and token_frase & parole_arresto:
            # & tra due set restituisce gli elementi in comune
            # la condizione è vera solo se SIA parole_siria SIA parole_arresto sono rappresentate nella frase
            frasi_trovate.append(frase)
    return frasi_trovate

df["cerca_occorenze_siria_arresto"] = df["full_text"].apply(lambda full_text: occorenza_siria_arresto(full_text, parole_siria, parole_arresto))
# applico la funzione a tutta la colonna full_text 
# il risultato è salvato in una nuova colonna: per ogni articolo, la lista delle frasi trovate

df["conferma_occorrenze_siria_arresto"] = df["cerca_occorenze_siria_arresto"].apply(lambda frasi_trovate: len(frasi_trovate) > 0)
# trasformo il risultato in True/False: True se l'articolo ha almeno una frase con entrambe le famiglie
print(df["conferma_occorrenze_siria_arresto"].sum())
# conta quanti articoli hanno almeno una frase con siria/variante vicino ad arresto/variante



Index(['headline', 'full_text', 'url'], dtype='str')
3


    frasi_trovate = []
    for frase in frasi: 
        token_frase = set(re.findall(r"\w+", frase.lower()))
        if token_frase & parole_siria and token_frase & parole_arresto:
            frasi_trovate.append(frase)
    return frasi_trovate

Per ogni frase dell'articolo, controlla se contiene almeno una parola della famiglia Siria e almeno una parola della famiglia arresto, se sì, salva quella frase intera nella lista dei risultati. Alla fine, restituisce tutte le frasi trovate in quell'articolo.

Ora cerco di fare la stessa cosa ma per tutte le parole che mi interessano. Scrivo un codice che mi dice in quante frasi le parole appaiono vicine.

In [ ]:
import re 
import pandas as pd

df = pd.read_csv("repubblica.csv")

print(df.columns)

def tokenizza(t):
    return re.findall(r"\w+", str(t).lower())

df["token"] = df["full_text"].apply(tokenizza) 

# set di nazionalità
parole_siria = {"siria", "siriano", "siriana", "siriani", "siriane"}
parole_afghanistan = {"afghanistan", "afghano", "afghana", "afghani", "afghane"}
parole_iraq = {"iraq", "iracheno", "irachena", "iracheni", "irachene"}
parole_libia = {"libia", "libico", "libica", "libici", "libiche"}
parole_sudan = {"sudan", "sudanese", "sudanesi"}

# set di parole "negative"
parole_arresto = {"arresto", "arresti", "arrestato", "arrestata", "arrestati", "arrestate"}
parole_crimine = {
    "reato", "reati", "crimine", "crimini", "criminale", "criminali",
    "illegale", "illegali", "clandestino", "clandestina", "clandestini", "clandestine",
    "irregolare", "irregolari",
    "espulsione", "espulsioni", "espulso", "espulsa", "espulsi", "espulse",
    "rimpatrio", "rimpatri", "rimpatriato", "rimpatriata", "rimpatriati", "rimpatriate",
    "denuncia", "denunce", "denunciato", "denunciata", "denunciati", "denunciate",
    "furto", "furti", "rapina", "rapine", "aggressione", "aggressioni", "stupro", "stupri", "spaccio",
    "carcere", "carceri", "detenuto", "detenuta", "detenuti", "detenute",
    "processo", "processi", "condanna", "condanne", "condannato", "condannata", "condannati", "condannate"
}
parole_violenza = {"violenza", "violenze", "violento", "violenta", "violenti", "violente"}
parole_polizia = {"polizia", "disordine", "carabinieri", "agenti"}
parole_immigrazione = {"immigrato", "immigrata", "immigrati", "immigrazione", "immigrate"}
parole_sicurezza = {
    "sicurezza", "disordine", "emergenza", "emergenze", "allarme", "allarmi", "minaccia", "minacce",
    "pericolo", "pericoli", "pericoloso", "pericolosa", "pericolosi", "pericolose", "invasione",
    "invasioni", "ondata", "ondate", "controllo", "controlli", "pattugliamento", "pattugliamenti"
}
parole_profugo = {
    "profugo", "profuga", "profughi", "profughe",
    "extracomunitario", "extracomunitaria", "extracomunitari", "extracomunitarie"
}

def co_occorrenza(full_text, set1, set2):
    frasi = re.split(r"(?<=[.!?])\s+", str(full_text))
    co_occ_trovate = []
    for frase in frasi:
        token_frase = set(re.findall(r"\w+", frase.lower())) # mette la frase in minuscolo, la tokenizza e trasforma le parole in un set
        if token_frase & set1 and token_frase & set2: # controlla che la frase contenga almeno una parola di set1 e almeno una di set2
            co_occ_trovate.append(frase) # se la condizione è vera, salva la frase originale (non i token) nella lista
    return co_occ_trovate
# questa è la funzione generica che posso applicare tutte le volte voglio 
# cambiando i due paramtri e il nome della colonna e usando .apply()

# creo due dizionari associando a ogni parola un valore che contiene i set scritti sopra
nazionalita = {
    "siria": parole_siria,
    "afghanistan": parole_afghanistan,
    "iraq": parole_iraq,
    "libia": parole_libia,
    "sudan": parole_sudan,
}

parole_negative = {
    "arresto": parole_arresto,
    "crimine": parole_crimine,
    "violenza": parole_violenza,
    "polizia": parole_polizia,
    "sicurezza": parole_sicurezza,
    "profugo": parole_profugo,
    "immigrazione": parole_immigrazione,
}


risultati = {} 
for nome_naz, set_naz in nazionalita.items():
    # .items() è un metodo dei dizionari che restituisce tutte le coppie chiave-valore per scorrerle con un for
    for nome_neg, set_neg in parole_negative.items():
# sono i for a creare le variabili nome_naz, set_naz, nome_neg, set_neg.
# le variabili nome_naz e _nome neg sono la parte a sinistra del dizionario, set_naz e set_neg sono a destra
# qui sto solo creando i nomi delle colonne dentro cui andranno i risulati
        colonna_frasi = f"frasi_{nome_naz}_{nome_neg}" # diventa per esempio frasi_siria_arresto
        
        df[colonna_frasi] = df["full_text"].apply(lambda t: co_occorrenza(t, set_naz, set_neg))
        # è la colonna in cui ogni cella contiene la lista delle frasi trovate in quell'articolo
        totale = (df[colonna_frasi].apply(len) > 0).sum()
        # .apply(len) mi dice quante frasi che contenevano le co-occorrenze sono state trovate in ciascun articolo
        # > 0 trasforma quei numeri in true false: true se c'è almeno una frase, false se non ce ne sono
        # .sum() conta i true
        risultati[f"{nome_naz} + {nome_neg}"] = totale
        print(f"{nome_naz} + {nome_neg}: {totale} articoli")

# riepilogo finale ordinato dal più frequente al meno frequente
print("\nRiepilogo ordinato")
# non posso prendere solo 'risultati' perchè prenderebbe TUTTO, anche le combinazioni di parole
# e le ordinerebbe alfabeticamente. key= serve per prendere solo i valori numerici usando .get

for combinazione in sorted(risultati, key=risultati.get, reverse=True):
    print(f"{combinazione}: {risultati[combinazione]}")


Index(['headline', 'full_text', 'url'], dtype='str')
siria + arresto: 96 articoli
siria + crimine: 335 articoli
siria + violenza: 139 articoli
siria + polizia: 107 articoli
siria + sicurezza: 577 articoli
siria + profugo: 336 articoli
siria + immigrazione: 55 articoli
afghanistan + arresto: 13 articoli
afghanistan + crimine: 80 articoli
afghanistan + violenza: 32 articoli
afghanistan + polizia: 40 articoli
afghanistan + sicurezza: 106 articoli
afghanistan + profugo: 34 articoli
afghanistan + immigrazione: 14 articoli
iraq + arresto: 50 articoli
iraq + crimine: 174 articoli
iraq + violenza: 85 articoli
iraq + polizia: 61 articoli
iraq + sicurezza: 405 articoli
iraq + profugo: 92 articoli
iraq + immigrazione: 14 articoli
libia + arresto: 36 articoli
libia + crimine: 277 articoli
libia + violenza: 98 articoli
libia + polizia: 57 articoli
libia + sicurezza: 407 articoli
libia + profugo: 62 articoli
libia + immigrazione: 120 articoli
sudan + arresto: 21 articoli
sudan + crimine: 91 articoli

Mi è venuto il dubbio che come altro segno di punteggiatura fosse necessario inserire anche le virgolette del discorso diretto e quindi ho voluto controllare se fossero presenti. 

In [6]:
import re 
import pandas as pd

df = pd.read_csv("repubblica.csv")

def virgolette(full_text):
    testo = str(full_text)
    if "«" in testo or "»" in testo:
        return True
    return False

df["ha_virgolette"] = df["full_text"].apply(virgolette)

print(df["ha_virgolette"].sum())




1647


Ma sono stupida e mi sono resa conto che non serve perchè la frase non si interrompe con il discorso diretto, se le persone non mi chiamassero mille volte fors enon perderei il filo di quello che sto facendo ogni dieci minuti